# 35B The agent graph
Five nodes, two routing functions, a repair edge with a cap, a budget guard and a trace. The kit runs the graph without LangGraph so you can see it; you wire the same nodes into a `StateGraph`.

In [1]:
import sys, json, pathlib, tempfile
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "askpanel").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from askpanel import audit, catalog, clock, db, econ, gate, llm, nodes, questions, standin, tools
P = yaml.safe_load(open(ROOT / "params.yaml"))
PANEL = db.load_panel(ROOT / P["data"]["panel"])
TMP = pathlib.Path(tempfile.mkdtemp())
BASE = db.build_db(PANEL, TMP / "base.sqlite")          # tables only: works before you write the view
print(f"{len(PANEL)} panel rows, {PANEL['adsh'].nunique()} filings; scratch database {BASE.name}")

74 panel rows, 4 filings; scratch database base.sqlite


In [2]:
led = audit.Ledger(clock=clock.clock)
T = tools.local_tools(BASE, led, tables=("filings", "holdings"))
N, after_check, after_execute = nodes.make_nodes(standin.FixtureSQLModel(), T, clock.clock,
                                                  allowed={"filings", "holdings"})
def ask(q, history=None):
    return nodes.run_plain(nodes.new_state(q, history), N, after_check, after_execute)
for q in ["How many current holdings does each BDC report?",
          "Which current debt holdings carry a PIK rate above 2%?",
          "Will Hotel Senior Lending cut its dividend?"]:
    s = ask(q)
    print(q, "\n ", " -> ".join(t["node"] + (f"[{t['outcome']}]" if t.get("outcome") else "") for t in s["trace"]))
    print("  ", s["answer"].replace("\n", "\n   "), "\n")

How many current holdings does each BDC report? 
  load_schema -> generate -> check[ok] -> execute[ok] -> summarize
   4 row(s)
   bdc | holdings
   FIXTURE GOLF CAPITAL CORP | 11
   FIXTURE HOTEL SENIOR LENDING | 8
   FIXTURE INDIA CREDIT FUND | 8
   FIXTURE JULIETT LENDING BDC | 10 

Which current debt holdings carry a PIK rate above 2%? 
  load_schema -> generate -> check[ok] -> execute[invalid] -> generate -> check[ok] -> execute[ok] -> summarize
   2 row(s)
   bdc | identifier | pik_rate
   FIXTURE HOTEL SENIOR LENDING | Oakmont Home Services LLC, First Lien Term Loan | 0.03
   FIXTURE JULIETT LENDING BDC | First Lien Term Loan | Foxglove Specialty Pharma Inc. | 03/2031 | 0.025 

Will Hotel Senior Lending cut its dividend? 
  load_schema -> generate -> check[decline] -> summarize
   I can only answer questions the BDC panel holds data for. 



In [3]:
s = ask("Which current debt holdings carry a PIK rate above 2%?")
pd.DataFrame(s["trace"]).drop(columns=["run_id"]).fillna("")

/var/folders/19/xhrx71p90m96kh792dr3hmlr0000gn/T/ipykernel_46941/4242910299.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  pd.DataFrame(s["trace"]).drop(columns=["run_id"]).fillna("")


,step,ts,node,schema_hash,attempt,request_key,usd,tokens_in,tokens_out,sql_hash,outcome,rows,result_hash,stop
0,1,2026-11-16T09:00:00-05:00,load_schema,e094ca8f543f,,,,,,,,,,
1,2,2026-11-16T09:00:00-05:00,generate,,0.0,c4d2c51263ad6948c301dc893fece5ba01febfe0a09515...,0.00128,2000.0,300.0,e54dbe96f2e0,,,,
2,3,2026-11-16T09:00:00-05:00,check,,,,,,,,ok,,,
3,4,2026-11-16T09:00:00-05:00,execute,,,,,,,,invalid,,,
4,5,2026-11-16T09:00:00-05:00,generate,,1.0,6ca3039384713d672f9daabbdd56856d2e998005954b0a...,0.00128,2000.0,300.0,8f3f9c91c165,,,,
5,6,2026-11-16T09:00:00-05:00,check,,,,,,,,ok,,,
6,7,2026-11-16T09:00:00-05:00,execute,,,,,,,,ok,2.0,2aefa1966bce,
7,8,2026-11-16T09:00:00-05:00,summarize,,,,,,,,,,,


## Record once, replay forever
The record/replay client from Lab 3B, in front of the model. Replay is free, offline and gives identical traces:

In [4]:
cache = TMP / "cache"
rec = llm.ReplayClient(cache, "record", call=standin.FixtureSQLModel())
rep = llm.ReplayClient(cache, "replay")
q = "Which current debt holdings carry a PIK rate above 2%?"
runs = []
for m in (rec, rep):
    n, a, b = nodes.make_nodes(m, T, clock.clock, allowed={"filings", "holdings"})
    runs.append(nodes.run_plain(nodes.new_state(q), n, a, b)["trace"])
print(len(list(cache.iterdir())), "cached responses; identical traces:", json.dumps(runs[0]) == json.dumps(runs[1]))

2 cached responses; identical traces: True


In [5]:
pricey = standin.FixtureSQLModel(usd=0.30)
n, a, b = nodes.make_nodes(pricey, T, clock.clock, usd_cap=0.10, allowed={"filings", "holdings"})
print(nodes.run_plain(nodes.new_state("How many current holdings does each BDC report?"), n, a, b)["answer"])

Stopped (budget_usd) before an answer.


## The same graph in LangGraph
Runs only if `langgraph` is installed. Note the explicit node names: the self-check reads them.

In [6]:
import importlib.util
if importlib.util.find_spec("langgraph") is None:
    print("langgraph not installed: skipped (pip install -r requirements.txt)")
else:
    try:
        from typing import Optional, TypedDict
        from langgraph.graph import StateGraph, START, END
        from langgraph.checkpoint.memory import InMemorySaver

        class AskState(TypedDict, total=False):
            question: str; history: list; schema: str; attempt: int; query: Optional[dict]
            error: Optional[dict]; result: Optional[dict]; answer: Optional[str]; trace: list
            usd: float; tokens: int; stop: Optional[str]; run_id: str

        g = StateGraph(AskState)
        g.add_node("load_schema", N["load_schema"]); g.add_node("generate", N["generate"])
        g.add_node("check", N["check"]); g.add_node("execute", N["execute"])
        g.add_node("summarize", N["summarize"])
        g.add_edge(START, "load_schema"); g.add_edge("load_schema", "generate"); g.add_edge("generate", "check")
        g.add_conditional_edges("check", after_check, {"execute": "execute", "generate": "generate", "summarize": "summarize"})
        g.add_conditional_edges("execute", after_execute, {"generate": "generate", "summarize": "summarize"})
        g.add_edge("summarize", END)
        app = g.compile(checkpointer=InMemorySaver())
        cfg = {"configurable": {"thread_id": "demo"}}
        s = app.invoke(nodes.new_state("Which current debt holdings carry a PIK rate above 2%?"), cfg)
        print(s["answer"])
        print("history kept by the checkpointer:", len(app.get_state(cfg).values["history"]), "turn(s)")
    except Exception as e:
        print("LangGraph demo failed:", type(e).__name__, e, "(check the version against the docs)")

langgraph not installed: skipped (pip install -r requirements.txt)


**Your turn (Part 3, 20 points).**
1. `agent/graph.py`: a `StateGraph` with the five nodes by name, conditional edges from `check` and `execute`, `compile(checkpointer=...)`. Run your Pydantic `validate_query` in the `check` node.
2. `agent/model.py`: a `ReplayClient` in front of Gemini (record mode needs `GEMINI_API_KEY`; replay needs nothing).
3. `agent/mcp_client.py`: the node tools call your server through the SDK v2 `Client`, never the database.
4. `agent/run.py --mode record|replay`: answers `questions/questions.yaml`, writes `outputs/questions/<id>.json`; a follow-up continues the thread of the question it follows.

**Implementation summary:** We implemented the five-node LangGraph workflow in `agent/graph.py`, with validation, limited retries, budget checks, and conversation history. `agent/model.py` connects Gemini through record/replay, `agent/mcp_client.py` calls the MCP server, and `agent/run.py` saves question results in `outputs/questions/`. The notebook uses a fake model to demonstrate the workflow without API costs.